# Data Preprocessing
<div class="alert alert-success">In this notebook, I implemented a data preprocessing pipeline that engineers new technical indicators, calculates price and volume returns, and performs feature normalization. The pipeline also handles missing values to ensure data integrity. The resulting datasets are stored in separate "train" and "test" directories. </div> 

<blockquote><b>Note:</b> To comply with licensing restrictions, the processed datasets have been excluded from this repository.</blockquote>

In [1]:
import pandas as pd
# To load the parquet file data into pandas dataframe
def load_parquet(file_path: str) -> pd.DataFrame:
    try:
        data = pd.read_parquet(file_path, engine='pyarrow')
        return data
    except Exception as e:
        print(f"An error occurred while importing data: {e}")
        return pd.DataFrame()



### Preprocess
Before implementing the training environment and DQN Agent, I should preprocess the stock prices to normalized (to generalize it) data values. 

<blockquote>According to the Markov Property, the Agent needs only the **current** state. To provide enough information as an observation, I chose to use RSI and MACD stock indicators. I didn't use SMA and EMA indicators as MACD itself a product of the EMA itself. I also wanted to use as few indicators as possible to improve the model performance.</blockquote> </br>

<div>🍁 SMA - Simple Moving Average  
Average of the stock price over the specified period  </div>
<div>🍁 EMA - Exponential Moving Average  
More current the data, it has more weight. It follows the price more closely. </div>
<div>🍁 RSI - Relative Strength Index </div>
<div>🍁 MACD - Moving Average Convergence/Divergence: Indicates bearish/bullish sentiment  </div>
</br>

$$
MACD = (\mathrm{EMA}_{\text{PRICE, 12\,period}} - \mathrm{EMA}_{\text{PRICE, 26\,period}})
$$

$$
\text{Signal Line} = \mathrm{EMA}_{\text{MACD, 9\,period}}
$$

Click this 👉 [link](https://www.fidelity.com/learning-center/trading-investing/technical-analysis/technical-indicator-guide/overview) for more information about technical indicators. 

#### Normalization
I used z-score for normalizing the MACD, Log-Volume feature values. </br>
**Z-Score**
$z=(x-\mu)/\sigma$


<blockquote>Changelog March 6, 2026: Noticed a bug in my code while training the model. I checked to eliminate the 0 prices below (Price 0 is wrong unless no one traded that stock that day.) but I used the greater than and equal to keep the 0s. Couple of files already have such issues and need to rerun the script with clean data for my next training with different hyperparameters. </blockquote>

In [2]:
import numpy as np
import ta

def preprocess_data(df: pd.DataFrame, window: int = 260) -> pd.DataFrame:
    df = df.copy()
    # skip all rows with missing values
    df = df.dropna(axis=0)

    # Step 1: Check for duplicates
    duplicate_rows = df.duplicated().sum()
    if duplicate_rows > 0:
        df = df.drop_duplicates()
    
    # Step 2: Check for integrity
    if (df[['Open', 'High', 'Low', 'Close', 'Volume']] <= 0).any().any():
        # only return the rows with positive prices
        valid_rows = (df[['Open', 'High', 'Low', 'Close', 'Volume']] > 0).all(axis=1)
        df = df[valid_rows]
    if (df['High'] < df['Low']).any():
        # only return the rows with High >= Low
        valid_rows = (df['High'] >= df['Low'])
        df = df[valid_rows]

    # Step 3: Calculate MACD
    macd =ta.trend.MACD(close=df['Close'], window_slow= 26, window_fast= 12, window_sign = 9, fillna = False) 
    df['MACD_line'] =macd.macd()
    df['MACD_signal'] =macd.macd_signal()
    df['MACD_diff'] =macd.macd_diff()

    # Step 4: Calculate RSI
    df['RSI']=ta.momentum.RSIIndicator(close= df['Close'], window= 14, fillna= False).rsi()

    # Step 5: Normalize the RSI, MACD, and Volume features
    #### RSI ####
    df['RSI_norm'] = (df['RSI'] - 50) / 50.0
    #### MACD ####
    mean_line = df['MACD_line'].rolling(window=window).mean()
    std_line = df['MACD_line'].rolling(window=window).std()
    mean_signal = df['MACD_signal'].rolling(window=window).mean()
    std_signal = df['MACD_signal'].rolling(window=window).std()
    mean_diff = df['MACD_diff'].rolling(window=window).mean()
    std_diff = df['MACD_diff'].rolling(window=window).std()
    df['MACD_line_norm'] = (df['MACD_line'] - mean_line) / (std_line + 1e-8) # adding epsilon to avoid division error
    df['MACD_signal_norm'] = (df['MACD_signal'] - mean_signal) / (std_signal + 1e-8)
    df['MACD_diff_norm'] = (df['MACD_diff'] - mean_diff) / (std_diff + 1e-8)
    #### Volume ####
    log_volume = np.log1p(df['Volume']) # Because volume ranges widely, using log() to manage the exponential growth. log1p is used to avoid log(0) issue.
    mean = log_volume.rolling(window=window).mean()
    std = log_volume.rolling(window=window).std()
    df['Volume_norm'] = (log_volume - mean) / (std + 1e-8)
    
    # Adding the previous close price as a feature
    df['Prev_close'] = df['Close'].shift(1)

    # Step 6: Add target price. This value is future price in 30 periods (days). 
    df['Target'] = df['Close'].shift(-30)

    # Step 7: Drop null value rows. The MACD, RSI ta calculations created many NaN value rows. Especially, the window size is as big as 260, I had to drop more than 1-year of data. 
    df = df.dropna(axis=0)

    # Step 8: Calculate log returns for the 'Close' price
    # There were 2 issues. 
    # 1. The .shift(1) created the first row with Prev_close as NaN. Solved in Step 6 by dropping null value rows.
    # 2. The log(0) issue. To avoid it, I am using .clip(lower=1e-8) method. 
    df['Return'] = np.log(df['Close'].clip(lower=1e-8)) - np.log(df['Prev_close'].clip(lower=1e-8))

    # Step 9: Drop unnecessary columns
    df = df.drop(columns=['Open', 'High', 'Low', 'Adj Close', 'Prev_close', 'MACD_line', 'MACD_signal', 'MACD_diff', 'RSI', 'Volume'])
    
    # Step 10: Return the preprocessed DataFrame
    return df

In [3]:
# test the function with apple stock data
train_file_path = 'parquet_train_data/AAPL.parquet'
test_file_path = 'parquet_test_data/AAPL.parquet'
df_aapl = load_parquet(train_file_path)
processed_df_aapl = preprocess_data(df_aapl)
df_aapl_test = load_parquet(test_file_path)
processed_df_aapl_test = preprocess_data(df_aapl_test)

In [4]:
display(processed_df_aapl.head())
display(processed_df_aapl_test.head())

,Close,RSI_norm,MACD_line_norm,MACD_signal_norm,MACD_diff_norm,Volume_norm,Target,Return
Date,,,,,,,,
1982-02-10,0.334821,-0.168715,0.052043,0.265932,-0.562678,0.536293,0.294643,0.013423
1982-02-11,0.330357,-0.203611,-0.014915,0.209773,-0.606959,-0.166255,0.290179,-0.013423
1982-02-12,0.334821,-0.151552,-0.039855,0.159323,-0.547999,-0.507200,0.296875,0.013423
1982-02-16,0.328125,-0.206961,-0.092100,0.107611,-0.565850,0.349158,0.301339,-0.020203
1982-02-17,0.332589,-0.152905,-0.103854,0.063683,-0.483898,-0.103271,0.301339,0.013514


,Close,RSI_norm,MACD_line_norm,MACD_signal_norm,MACD_diff_norm,Volume_norm,Target,Return
Date,,,,,,,,
2021-06-01,124.279999,-0.162555,-1.131932,-1.185974,-0.128403,-1.502697,149.149994,-0.002652
2021-06-02,125.059998,-0.112683,-1.118775,-1.190792,-0.076224,-1.843607,148.479996,0.006257
2021-06-03,123.540001,-0.185968,-1.152905,-1.202003,-0.146132,-1.156407,146.389999,-0.012229
2021-06-04,125.889999,-0.042596,-1.092184,-1.197659,0.021889,-1.192238,142.449997,0.018844
2021-06-07,125.900002,-0.042018,-1.037881,-1.182311,0.141948,-1.333705,146.149994,0.000079


## Pre-processing datasets

Now I have my script ready with the `preprocess_data()` function, and using that I will preprocess all *.parquet files using the function `data_process()`. 

In [9]:
from pathlib import Path
from tqdm import tqdm
Path("dataset_train").mkdir(parents=True, exist_ok=True)
Path("dataset_validation").mkdir(parents=True, exist_ok=True)

def data_process(folder: str) -> pd.DataFrame:

    files = Path(folder).glob("*.parquet")
    for file in tqdm(files):
        try:
            data = pd.read_parquet(file, engine='pyarrow')
            processed_data = preprocess_data(data)
            if 'train' in folder:
                processed_data.to_parquet(f'dataset_train/{file.stem}.parquet', engine='pyarrow', compression='snappy')
            elif 'test' in folder:
                processed_data.to_parquet(f'dataset_validation/{file.stem}.parquet', engine='pyarrow', compression='snappy')
        except Exception as e:
            print(f"An error occurred while processing {file}: {e}")

In [8]:
# processing the training data
data_process('parquet_train_data')

3940it [00:42, 93.74it/s] 


In [15]:
# processing the validation data
data_process('parquet_test_data')

3940it [00:32, 122.10it/s]


In [18]:
# test the new dataset for apple stock
processed_asys = pd.read_parquet('dataset_train/ASYS.parquet', engine='pyarrow')
display(processed_asys.head())

,Close,RSI_norm,MACD_line_norm,MACD_signal_norm,MACD_diff_norm,Volume_norm,Target,Return
Date,,,,,,,,
1994-11-07,3.50000,0.058388,-0.167666,-0.043697,-0.360762,0.964487,4.2500,0.036368
1994-11-10,3.53125,0.079457,-0.146163,-0.066568,-0.239042,0.691725,4.6250,0.008889
1994-11-14,3.75000,0.212321,-0.013204,-0.056235,0.110009,0.779939,4.4375,0.060104
1994-11-15,3.62500,0.113427,0.017780,-0.041269,0.157899,-0.256019,4.5000,-0.033902
1994-11-16,3.75000,0.185022,0.105643,-0.010300,0.323856,0.465275,4.2500,0.033902


In [19]:
processed_asys[processed_asys['Close'] == 0]

,Close,RSI_norm,MACD_line_norm,MACD_signal_norm,MACD_diff_norm,Volume_norm,Target,Return
Date,,,,,,,,


In [20]:
processed_asys_val = pd.read_parquet('dataset_validation/ASYS.parquet', engine='pyarrow')
display(processed_asys_val.head())

,Close,RSI_norm,MACD_line_norm,MACD_signal_norm,MACD_diff_norm,Volume_norm,Target,Return
Date,,,,,,,,
2021-06-01,10.31,0.092155,-0.890005,-1.713246,2.033958,0.395373,9.75,-0.012530
2021-06-02,10.11,0.034523,-0.781303,-1.535527,1.876636,0.727449,9.35,-0.019589
2021-06-03,9.76,-0.059052,-0.780067,-1.394278,1.495557,0.277988,9.08,-0.035233
2021-06-04,9.83,-0.038814,-0.757123,-1.277030,1.246099,-0.044778,8.88,0.007147
2021-06-07,9.80,-0.047217,-0.742507,-1.180546,1.028132,0.380309,9.35,-0.003057


In [22]:
print("\n\U0001F422 ASYS Training Data Shape:", processed_aapl.shape)
print("\n🐢 ASYS Validation Data Shape:", processed_aapl_val.shape)


🐢 ASYS Training Data Shape: (6289, 8)

🐢 ASYS Validation Data Shape: (1143, 8)


## Conclusion
🌴 The data preprocessing is complete.  
🌴 Ready for training RL now.  


|Date (YYYY-MM-DD)|Version|Created By|  
|--|--|--|
|2026-02-26|1.0|Battogtokh Baasanjav|
|2026-03-06|1.1|Battogtokh Baasanjav|